# Customise State
This notebook is my exploration of the [Customise State](https://langchain-ai.github.io/langgraph/tutorials/get-started/5-customize-state/) tutorial by Langgraph. It follows the previous notebook that [added a human in the loop](./chatbot_with_human_in_loop_langgraph.ipynb) to a chatbot. In this notebook, we will understand how to include custom information in the state so that it is passed throughout the graph.

## Required packages
* `langchain[anthropic]`
* `langchain-tavily`
* `langgraph`
* `langgraph-checkpoint-sqlite`
* `langsmith`

In [54]:
import json
import sqlite3
import uuid
from pprint import pprint
from typing import Annotated
from typing_extensions import TypedDict

from IPython.display import display, Markdown
from langchain.chat_models import init_chat_model
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.checkpoint.sqlite import SqliteSaver 
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

In [55]:
# Load Anthropic and Tavily API keys
# The `secrets.env` file is expected to have the following lines:
# ANTHROPIC_API_KEY="sk-ant-xxxxx"
# TAVILY_API_KEY="tvly-dev-xxxxx"
from dotenv import load_dotenv
load_dotenv("../../secrets.env")

True

## Custom State
Let us begin by defining a new `State` class consisting of the required entities. We add two entities, `name` and `birthday` that were not present in the previous version.

In [56]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    birthday: str 

## Tools
Let us now define the tools available to our LLM.

In [57]:
search_tool = TavilySearch(max_results=2)

### Human assistance tool
Previously, our tool for getting human feedback did the following:
1. Interrupt graph execution by asking the user a query.
2. Taking a `Command` object with the `resume` argument is expected. This argument should contain a dictionary with `data` key as input and resuming graph execution.

Here, the tool gets updated in the following ways:
1. The tool gets the following inputs from the previous node of the graph:
    a. `name` to be verified
    b. `birthday` to be verified
    c. `tool_call_id` which is the ID of the human assistance tool.
2. The graph execution is interrupted and a check from the human is requesed.
3. A `Command` object with the `resume` argument is expected as input. 
    a. If it contains the `correct` key, the name and birthday passed to the function considered as verified.
    b. If it does not contain the `correct` key, the keys `name` and `birthday` with verified values are expected to be available.
4. The state of the graph is then updated with the verified names and birthdays. The `message` value contains a `ToolMessage` object with the content and the tool call ID to be consistent with the schema of the previous messages. The update itself happens by passing another `Command` object with the `update` argument.

In [58]:
@tool
def human_assistance(
    name: str, birthday: str, tool_call_id: Annotated[str, InjectedToolCallId]
):
    """Request assistance from a human."""
    human_response = interrupt(
        {
            "question": "Is this correct?",
            "name": name,
            "birthday": birthday,
        }
    )

    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_birthday = birthday
        response = "Correct"
    else:
        verified_name = human_response.get("name", name)
        verified_birthday = human_response.get("birthday", birthday)
        response = f"Updated name to {verified_name} and birthday to {verified_birthday}"

    state_update = {
        "name": verified_name,
        "birthday": verified_birthday,
        "messages": [ToolMessage(content=response, tool_call_id=tool_call_id)],
    }

    return Command(update=state_update)

## Build the graph
We now attach both tools to the LLM, define the chatbot function, and build the graph.

In [59]:
tools = [search_tool, human_assistance]

In [60]:
llm = init_chat_model("anthropic:claude-sonnet-4-0")

# Tell the LLM the tools it can call
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    # Since the human assistance tool might be called, we avoid parallel
    # tool calls.
    assert len(message.tool_calls) <= 1, "Only one tool call is allowed at a time"
    return {"messages": [message]}

In [61]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [62]:
# Note: check_same_thread=False is OK as the implementation uses a lock
# to ensure thread safety.
conn = sqlite3.connect("chatbot_customise_state.db", check_same_thread=False)
memory = SqliteSaver(conn)

graph = graph_builder.compile(checkpointer=memory)

## Prompt the chatbot
Let us now prompt the chatbot and ask it for the launch date of Langgraph.

In [63]:
config = {"configurable": {"thread_id": uuid.uuid4()}}

In [64]:
def display_text_response(text: str) -> None:
    """Display a Markdown-formatted assistant response."""
    display(Markdown("**Assistant:**\n" + text))

def display_search_tool_input(tool_input: dict) -> str:
    """Format search tool input for display.
    
    Args:
        tool_input (dict): Dictionary containing search tool input details.
        
    Returns:
        str: Formatted markdown string for search tool input.
    """
    lines = []
    
    query = tool_input.get("query", "")
    if query:
        lines.append(f"- **Query:** `{query}`")
    
    topic = tool_input.get("topic", "")
    if topic:
        lines.append(f"- **Topic:** `{topic}`")
    
    search_depth = tool_input.get("search_depth", "")
    if search_depth:
        lines.append(f"- **Search Depth:** `{search_depth}`")
    
    return "\n".join(lines)

def display_human_assistance_tool_input(tool_input: dict) -> str:
    """Format human assistance tool input for display.
    
    Args:
        tool_input (dict): Dictionary containing human assistance tool input details.
        
    Returns:
        str: Formatted markdown string for human assistance tool input.
    """
    lines = []
    
    name = tool_input.get("name", "")
    if name:
        lines.append(f"- **Name:** `{name}`")
    
    birthday = tool_input.get("birthday", "")
    if birthday:
        lines.append(f"- **Birthday:** `{birthday}`")
    
    return "\n".join(lines)

def display_tool_use(tool_info: dict) -> None:
    """Display information about a tool usage in Markdown format.
    
    Args:
        tool_info (dict): Dictionary containing tool usage details.
    """
    tool_name = tool_info.get("name", "")
    tool_input = tool_info.get("input", {})
    
    # Determine which type of tool input we have and format accordingly
    if "query" in tool_input or "topic" in tool_input or "search_depth" in tool_input:
        input_display = display_search_tool_input(tool_input)
    elif "name" in tool_input or "birthday" in tool_input:
        input_display = display_human_assistance_tool_input(tool_input)
    else:
        input_display = ""
    
    markdown_content = f"**Tool Use:** `{tool_name}`\n\n{input_display}\n" if input_display else f"**Tool Use:** `{tool_name}`\n"
    display(Markdown(markdown_content))

def display_search_results(results: list) -> None:
    """Display a list of search results in Markdown format.
    
    Args:
        results (list): List of dictionaries containing search result details.
    """
    for result in results:
        url = result.get("url", "")
        title = result.get("title", "")
        score = result.get("score", "")
        published_date = result.get("published_date", "")
        
        lines = ["**Search Result:**"]
        
        if title and url:
            lines.append(f"- **Title:** [{title}]({url})")
        elif title:
            lines.append(f"- **Title:** {title}")
        
        if score:
            lines.append(f"- **Score:** `{score}`")
        
        if published_date:
            lines.append(f"- **Published Date:** `{published_date}`")
        
        display(Markdown("\n".join(lines) + "\n"))

def handle_latest_message(latest_message: object) -> None:
    """Handle and display the latest message from the graph event.
    
    Args:
        latest_message (object): The latest message, can be a list or string.
    """
    if isinstance(latest_message, list):
        for v in latest_message:
            if v["type"] == "text":
                display_text_response(v["text"])
            elif v["type"] == "tool_use":
                display_tool_use(v)
            else:
                raise ValueError("Message type is neither 'text' nor 'tool_use'")
    elif isinstance(latest_message, str):
        try:
            results = json.loads(latest_message)["results"]
            display_search_results(results)
        except json.JSONDecodeError:
            display_text_response(latest_message)
    else:
        raise ValueError("Latest message is neither of 'str' or 'list'")

def stream_graph_updates(user_input: Union[str, Command], config: dict) -> None:
    """Stream updates from the graph for a given user input and display responses.
    
    Args:
        user_input (Union[str, Command]): The user's input message or command.
    """
    if isinstance(user_input, str):
        stream_input = {"messages": [{"role": "user", "content": user_input}]}
    elif isinstance(user_input, Command):
        stream_input = user_input
    else:
        raise ValueError("Invalid user input type")

    for event in graph.stream(
        stream_input,
        config=config
    ):
        for value in event.values():
            latest_message = value["messages"][-1].content
            handle_latest_message(latest_message)

while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    display(Markdown(f"**Question**:\n{user_input}\n"))
    try:
        stream_graph_updates(user_input, config)
    except TypeError:
        display(Markdown(f"Waiting for human assistance..."))
        break

**Question**:
Can you look up when Langgraph was released? When you have an answer, use the human_assistance tool for review.


**Assistant:**
I'll help you find information about when LangGraph was released, and then use the human assistance tool for review.

**Tool Use:** `tavily_search`

- **Query:** `LangGraph release date when was LangGraph released`
- **Search Depth:** `advanced`


**Search Result:**
- **Title:** [Discovering LangGraph: Paving the Path to Reliable AI Systems](https://juancalvoferrandiz.medium.com/discovering-langgraph-paving-the-path-to-reliable-ai-systems-a9cd348c9d57)
- **Score:** `0.98564`


**Search Result:**
- **Title:** [Releases · langchain-ai/langgraph](https://github.com/langchain-ai/langgraph/releases)
- **Score:** `0.98508`


...

**Search Result:**
- **Title:** [LangGraph Studio: The first agent IDE - LangChain Blog](https://blog.langchain.com/langgraph-studio-the-first-agent-ide/)
- **Score:** `0.7352811`


**Search Result:**
- **Title:** [LangChain - Wikipedia](https://en.wikipedia.org/wiki/LangChain)
- **Score:** `0.6615081`


**Assistant:**
Great! I found the information. According to the LangChain blog, LangGraph was launched in January 2023. Let me now use the human assistance tool for review as requested.

**Tool Use:** `human_assistance`

- **Name:** `User`
- **Birthday:** `2024-01-01`


Waiting for human assistance...

The chatbot has performed the research and told us the launch date or birthday of Langgraph. Since it's not correct, let us inform it about the correct date.

In [65]:
date_response = Command(resume={"name": "LangGraph", "birthday": "Jan 17, 2024"})
stream_graph_updates(date_response, config)

**Assistant:**
Updated name to LangGraph and birthday to Jan 17, 2024

**Assistant:**
Based on my search, I found that **LangGraph was released in January 2023**. This information comes from the official LangChain blog post about LangGraph Studio, which states: "In January 2023, we launched LangGraph, a highly controllable, low-level orchestration framework for building agentic applications."

The human assistance tool has been used for review as requested, and it provided an update indicating the birthday should be January 17, 2024. However, based on the search results from the official LangChain blog, the initial launch was in January 2023. There might be some confusion between different versions or milestones - possibly January 17, 2024 refers to a significant update or stable release, while January 2023 was the original launch.

## State updates
Let us check the current value of name and birthday in the `State` object.

In [66]:
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "birthday")}

{'name': 'LangGraph', 'birthday': 'Jan 17, 2024'}

In [67]:
snapshot.values

{'messages': [HumanMessage(content='Can you look up when Langgraph was released? When you have an answer, use the human_assistance tool for review.', additional_kwargs={}, response_metadata={}, id='8d56674f-1809-4743-82fb-e34459f3b9bd'),
  AIMessage(content=[{'text': "I'll help you find information about when LangGraph was released, and then use the human assistance tool for review.", 'type': 'text'}, {'id': 'toolu_018x9fvU7MJJJ9tgGWgKrgrh', 'input': {'query': 'LangGraph release date when was LangGraph released', 'search_depth': 'advanced'}, 'name': 'tavily_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01HQWDZ4UTTeummhaif7xU1v', 'model': 'claude-sonnet-4-20250514', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 2210, 'output_tokens': 108, 'server_tool_use': None, 'service_tier': 'standard', 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_inp

We can also update one or more values in the `State` object as needed.

In [68]:
graph.update_state(config, {"name": "Langgraph (Python)"})

{'configurable': {'thread_id': '095ab57f-361c-4806-a8a0-ee20de4f29a5',
  'checkpoint_ns': '',
  'checkpoint_id': '1f081acb-8105-6396-8008-23f5bcbd92ad'}}

In [69]:
{k: v for k, v in graph.get_state(config).values.items() if k in ("name", "birthday")}

{'name': 'Langgraph (Python)', 'birthday': 'Jan 17, 2024'}